Aggregation means summarizing data:

Example:

Total sales
Average Revenue
Frequency counts

In [30]:
#GroupBy
import pandas as pd

data = {
    "Customer": ["Asha","Bala","Chitra","Deepa","Esha","Farhan","Gita","Hari"],
    "City": ["Bangalore","Delhi","Mumbai","Delhi","Bangalore","Mumbai","Delhi","Bangalore"],
    "Product": ["Laptop","Mobile","Laptop","Tablet","Mobile","Laptop","Tablet","Mobile"],
    "Sales": [50000,30000,45000,20000,32000,47000,22000,31000],
    "Quantity": [2,1,1,3,2,1,2,1]
}

df = pd.DataFrame(data)

df

,Customer,City,Product,Sales,Quantity
0,Asha,Bangalore,Laptop,50000,2
1,Bala,Delhi,Mobile,30000,1
2,Chitra,Mumbai,Laptop,45000,1
3,Deepa,Delhi,Tablet,20000,3
4,Esha,Bangalore,Mobile,32000,2
5,Farhan,Mumbai,Laptop,47000,1
6,Gita,Delhi,Tablet,22000,2
7,Hari,Bangalore,Mobile,31000,1


In [3]:
#Group by works with split,apply,combine logic
#Total Sales by city
df.groupby("City")["Sales"].sum()


City
Bangalore    113000
Delhi         72000
Mumbai        92000
Name: Sales, dtype: int64

In [12]:
#multi-index (Hierarchial index)
df.groupby(["City","Product"])["Sales"].sum()

City       Product
Bangalore  Laptop     50000
           Mobile     63000
Delhi      Mobile     30000
           Tablet     42000
Mumbai     Laptop     92000
Name: Sales, dtype: int64

In [ ]:
#count transactions per city
df.groupby("City")["Sales"].count()

City
Bangalore    3
Delhi        3
Mumbai       2
Name: Sales, dtype: int64

In [5]:
#Sales by city and product
df.groupby(["City","Product"])["Sales"].sum()

City       Product
Bangalore  Laptop     50000
           Mobile     63000
Delhi      Mobile     30000
           Tablet     42000
Mumbai     Laptop     92000
Name: Sales, dtype: int64

In [7]:
df.groupby(["Product","City"])["Sales"].sum()

Product  City     
Laptop   Bangalore    50000
         Mumbai       92000
Mobile   Bangalore    63000
         Delhi        30000
Tablet   Delhi        42000
Name: Sales, dtype: int64

In [8]:
#multiple aggregations
#using .agg()

df.groupby("City").agg({
    "Sales":"sum",
    "Quantity":"mean"
})

,Sales,Quantity
City,,
Bangalore,113000,1.666667
Delhi,72000,2.000000
Mumbai,92000,1.000000


In [9]:
#multiple aggregations on same column

df.groupby("City")["Sales"].agg(["sum","mean","max","min"])

,sum,mean,max,min
City,,,,
Bangalore,113000,37666.666667,50000,31000
Delhi,72000,24000.000000,30000,20000
Mumbai,92000,46000.000000,47000,45000


In [17]:
#custom aggregation
df=pd.DataFrame({
    "Category":["A","A","A","B","B"],
    "Sales":[100,150,200,300,250]
})

df.groupby("Category")["Sales"].agg(lambda x:x.max()-x.min())

#range=max-min

Category
A    100
B     50
Name: Sales, dtype: int64

In [ ]:
#group filtering

df.groupby("Category").filter(lambda x:x["Sales"].mean()>150)

#keeps groups whose average is greater than 150

,Category,Sales
3,B,300
4,B,250


In [ ]:
#Null values are dropped
df=pd.DataFrame({"City":["Delhi","Mumbai",None],
                 "Sales":[100,200,300]})

df.groupby("City").sum()

,Sales
City,
Delhi,100
Mumbai,200


In [ ]:
#groupby size vs count

df.groupby("City").size()   #counts number of rows per group

df.groupby("City")["Sales"].count()  #counts non null values in sales column

City
Bangalore    3
Delhi        3
Mumbai       2
Name: Sales, dtype: int64

In [12]:
#pivot tables
#Pivot tables reshape data like excel pivot tables

pd.pivot_table(df,values="Sales",
                  index="City",
                  columns="Product",
                  aggfunc="sum")

Product,Laptop,Mobile,Tablet
City,,,
Bangalore,50000.0,63000.0,NaN
Delhi,NaN,30000.0,42000.0
Mumbai,92000.0,NaN,NaN


In [13]:
#pivot table with multiple aggregations
pd.pivot_table(df,values="Sales",
                  index="City",
                  columns="Product",
                  aggfunc=["sum","mean"])

sum                       mean                  
Product     Laptop   Mobile   Tablet   Laptop   Mobile   Tablet
City                                                           
Bangalore  50000.0  63000.0      NaN  50000.0  31500.0      NaN
Delhi          NaN  30000.0  42000.0      NaN  30000.0  21000.0
Mumbai     92000.0      NaN      NaN  46000.0      NaN      NaN

In [ ]:
#crosstab
#crosstab counts occurrences it's also called frequency table

pd.crosstab(df["City"],df["Product"])

#This gives us product popularity in each city

Product,Laptop,Mobile,Tablet
City,,,
Bangalore,1,2,0
Delhi,0,1,2
Mumbai,2,0,0


In [3]:
#pivot table with multiple values

pd.pivot_table(df,values=["Sales","Quantity"],
               index="City",
               columns="Product",
               aggfunc="sum")

Quantity                  Sales                  
Product     Laptop Mobile Tablet   Laptop   Mobile   Tablet
City                                                       
Bangalore      2.0    3.0    NaN  50000.0  63000.0      NaN
Delhi          NaN    1.0    5.0      NaN  30000.0  42000.0
Mumbai         2.0    NaN    NaN  92000.0      NaN      NaN

In [ ]:
#pivot table with fill values
pd.pivot_table(df,values="Sales",
               index="City",
               columns="Product",
               aggfunc="sum",
               fill_value=0)

#replace NaN with zero

Product,Laptop,Mobile,Tablet
City,,,
Bangalore,50000,63000,0
Delhi,0,30000,42000
Mumbai,92000,0,0


In [ ]:
#pivot table with margins
pd.pivot_table(df,values="Sales",
                  index="City",
                  columns="Product",
                  aggfunc=["sum","mean"],
                  margins=True)

#gives all row with totals (subtotal,grandtotal)

sum                                    mean                    \
Product      Laptop   Mobile   Tablet     All        Laptop   Mobile   Tablet   
City                                                                            
Bangalore   50000.0  63000.0      NaN  113000  50000.000000  31500.0      NaN   
Delhi           NaN  30000.0  42000.0   72000           NaN  30000.0  21000.0   
Mumbai      92000.0      NaN      NaN   92000  46000.000000      NaN      NaN   
All        142000.0  93000.0  42000.0  277000  47333.333333  31000.0  21000.0   

                         
Product             All  
City                     
Bangalore  37666.666667  
Delhi      24000.000000  
Mumbai     46000.000000  
All        34625.000000

In [15]:
import pandas as pd

data = {
    "Date": [
        "2024-01-01","2024-01-01",
        "2024-01-02","2024-01-02",
        "2024-01-03"
    ],
    "Revenue": [10000, 15000, 20000, 12000, 18000],
    "Margin": [0.20, 0.25, 0.30, 0.28, 0.22]
}

df = pd.DataFrame(data)

df

,Date,Revenue,Margin
0,2024-01-01,10000,0.20
1,2024-01-01,15000,0.25
2,2024-01-02,20000,0.30
3,2024-01-02,12000,0.28
4,2024-01-03,18000,0.22


In [16]:
#Named aggregation

df.groupby("Date").agg(Total_revenue=("Revenue","sum"),
                       Avg_margin=("Margin","mean"))

,Total_revenue,Avg_margin
Date,,
2024-01-01,25000,0.225
2024-01-02,32000,0.290
2024-01-03,18000,0.220


In [26]:
import pandas as pd

data = {
    "Salesperson": ["Asha","Asha","Asha","Bala","Bala","Chitra"],
    "Date": [
        "2024-01-01","2024-01-02","2024-01-03",
        "2024-01-01","2024-01-02","2024-01-01"
    ],
    "Amount": [1000,1200,800,1500,1700,2000]
}

df = pd.DataFrame(data)

df

,Salesperson,Date,Amount
0,Asha,2024-01-01,1000
1,Asha,2024-01-02,1200
2,Asha,2024-01-03,800
3,Bala,2024-01-01,1500
4,Bala,2024-01-02,1700
5,Chitra,2024-01-01,2000


In [27]:
#to calculate average sales 

df.groupby("Salesperson").agg({"Amount":"mean"})

,Amount
Salesperson,
Asha,1000.0
Bala,1600.0
Chitra,2000.0


In [ ]:
#using transform

df["avg_sales"]=df.groupby("Salesperson")["Amount"].transform("mean")
df

#transform returns same number of rows 
#groupby reduces rows

,Salesperson,Date,Amount,avg_sales
0,Asha,2024-01-01,1000,1000.0
1,Asha,2024-01-02,1200,1000.0
2,Asha,2024-01-03,800,1000.0
3,Bala,2024-01-01,1500,1600.0
4,Bala,2024-01-02,1700,1600.0
5,Chitra,2024-01-01,2000,2000.0


In [33]:
import pandas as pd

data = {
    "Category": [
        "Electronics","Electronics","Electronics","Electronics",
        "Furniture","Furniture","Furniture","Furniture"
    ],
    "Product": [
        "Laptop","Mobile","Tablet","Camera",
        "Chair","Table","Sofa","Bed"
    ],
    "Sales": [50000,65000,30000,45000,20000,25000,40000,35000]
}

df = pd.DataFrame(data)

df

,Category,Product,Sales
0,Electronics,Laptop,50000
1,Electronics,Mobile,65000
2,Electronics,Tablet,30000
3,Electronics,Camera,45000
4,Furniture,Chair,20000
5,Furniture,Table,25000
6,Furniture,Sofa,40000
7,Furniture,Bed,35000


In [ ]:
#finding top N per group
#for example top 3 products per category

df.sort_values("Sales",ascending=False).groupby("Category").head(3)

,Category,Product,Sales
1,Electronics,Mobile,65000
0,Electronics,Laptop,50000
3,Electronics,Camera,45000
6,Furniture,Sofa,40000
7,Furniture,Bed,35000
5,Furniture,Table,25000


In [35]:
import pandas as pd

data = {
    "Branch": ["North","North","South","South","East","East"],
    "Month": ["Jan","Feb","Jan","Feb","Jan","Feb"],
    "Sales": [10000,12000,8000,9000,7000,9500]
}

df = pd.DataFrame(data)

df

,Branch,Month,Sales
0,North,Jan,10000
1,North,Feb,12000
2,South,Jan,8000
3,South,Feb,9000
4,East,Jan,7000
5,East,Feb,9500


In [37]:
#calculate percentage change in sales per branch

df.groupby("Branch")["Sales"].sum().pct_change()

Branch
East          NaN
North    0.333333
South   -0.227273
Name: Sales, dtype: float64

In [38]:
df["Sales"].pct_change()

0         NaN
1    0.200000
2   -0.333333
3    0.125000
4   -0.222222
5    0.357143
Name: Sales, dtype: float64